In [2]:
import json
import pandas as pd
from datasets import load_dataset

#load the filtered hotel data we saved in Phase 1
import os
os.getcwd()

c:\Users\Hamza\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'c:\\Users\\Hamza\\hotel-dialogue-system\\notebooks'

In [3]:
# Load the filtered hotel training data we saved in Phase 1
hotel_train = pd.read_json("../data/multiwoz/hotel_train.json", lines=True)
hotel_val = pd.read_json("../data/multiwoz/hotel_val.json", lines=True)

print(f"Train: {len(hotel_train)} dialogues")
print(f"Val: {len(hotel_val)} dialogues")
hotel_train.head()

Train: 3369 dialogues
Val: 418 dialogues


,dialogue_id,services,turns
0,PMUL4398.json,"[restaurant, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
1,SNG1013.json,[hotel],"{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
2,PMUL0121.json,"[restaurant, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
3,PMUL3484.json,"[restaurant, taxi, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."
4,PMUL2979.json,"[taxi, attraction, hotel]","{'turn_id': ['0', '1', '2', '3', '4', '5', '6'..."


In [4]:
# Extract user turns with hotel intents
rows = []

for _, dialogue in hotel_train.iterrows():
    turns = dialogue["turns"]
    
    for i, (speaker, utterance, frames) in enumerate(zip(
        turns["speaker"], turns["utterance"], turns["frames"]
    )):
        # Only look at user turns (speaker == 0)
        if speaker != 0:
            continue
        
        # Check each frame for hotel intent
        for j, state in enumerate(frames["state"]):
            intent = state["active_intent"]
            service = frames["service"][j]
            
            if service == "hotel" and intent in ["find_hotel", "book_hotel"]:
                rows.append({
                    "utterance": utterance,
                    "intent": intent,
                    "dialogue_id": dialogue["dialogue_id"]
                })

df = pd.DataFrame(rows)
print(f"Total hotel turns: {len(df)}")
print(df["intent"].value_counts())
df.head()

Total hotel turns: 14868
intent
find_hotel    10465
book_hotel     4403
Name: count, dtype: int64


,utterance,intent,dialogue_id
0,i need a place to dine in the center thats exp...,find_hotel,PMUL4398.json
1,"Any sort of food would be fine, as long as it ...",find_hotel,PMUL4398.json
2,"Sounds good, could I get that phone number? Al...",find_hotel,PMUL4398.json
3,Yes. Can you book it for me?,find_hotel,PMUL4398.json
4,i want to book it for 2 people and 2 nights st...,book_hotel,PMUL4398.json


In [5]:
# Drop sentences that are too short or clearly not about hotels
df = df[df["utterance"].str.split().str.len() > 3]

# Shuffle the data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"After cleaning: {len(df)} turns")
print(df["intent"].value_counts())
df.head()

After cleaning: 14723 turns
intent
find_hotel    10389
book_hotel     4334
Name: count, dtype: int64


,utterance,intent,dialogue_id
0,Is that a 2 star hotel?,find_hotel,MUL0767.json
1,Is it in the north? I need to be in the north ...,find_hotel,SSNG0284.json
2,"Great, and what is the address? Also, can I ju...",find_hotel,MUL2528.json
3,I want to to have a 4 star rating & be a guest...,find_hotel,PMUL3849.json
4,How about a different hotel in the same price ...,book_hotel,SSNG0349.json


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    df["utterance"], df["intent"],
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 11778
Test samples: 2945


In [7]:
# Step 1 — convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 2 — train logistic regression classifier
classifier = LogisticRegression(class_weight="balanced", max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

print("Model trained.")

Model trained.


In [8]:
# Evaluate the classifier
y_pred = classifier.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

  book_hotel       0.78      0.86      0.82       860
  find_hotel       0.94      0.90      0.92      2085

    accuracy                           0.89      2945
   macro avg       0.86      0.88      0.87      2945
weighted avg       0.89      0.89      0.89      2945



In [9]:
# Test on custom sentences
test_sentences = [
    "I need a cheap hotel in the centre",
    "Can you book it for 2 nights for 3 people",
    "I am looking for a guesthouse with free parking",
    "Please make a reservation for monday"
]

# Vectorise and predict
X_custom = vectorizer.transform(test_sentences)
predictions = classifier.predict(X_custom)

for sentence, prediction in zip(test_sentences, predictions):
    print(f"'{sentence}'\n  → {prediction}\n")

'I need a cheap hotel in the centre'
  → find_hotel

'Can you book it for 2 nights for 3 people'
  → book_hotel

'I am looking for a guesthouse with free parking'
  → find_hotel

'Please make a reservation for monday'
  → book_hotel



In [10]:
import pickle
import os

os.makedirs("../data/models", exist_ok=True)

# Save the vectoriser and classifier
with open("../data/models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("../data/models/intent_classifier.pkl", "wb") as f:
    pickle.dump(classifier, f)

print("Models saved to data/models/")

Models saved to data/models/


In [12]:
# nlu.py
# Natural Language Understanding module.
# Takes raw user input and returns the intent and extracted slots.

import pickle
import os

class IntentClassifier:
    def __init__(self, model_path="../data/models/intent_classifier.pkl",
                 vectorizer_path="../data/models/tfidf_vectorizer.pkl"):
        # Load the saved classifier and vectoriser from disk
        with open(model_path, "rb") as f:
            self.classifier = pickle.load(f)
        with open(vectorizer_path, "rb") as f:
            self.vectorizer = pickle.load(f)

    def predict(self, text):
        # Convert text to TF-IDF vector and predict intent
        X = self.vectorizer.transform([text])
        intent = self.classifier.predict(X)[0]
        confidence = self.classifier.predict_proba(X).max()
        return {
            "intent": intent,
            "confidence": round(float(confidence), 2)
        }


if __name__ == "__main__":
    # Quick test when running this file directly
    clf = IntentClassifier()
    test = "I need a cheap hotel in the north"
    print(clf.predict(test))

{'intent': 'find_hotel', 'confidence': 0.92}


In [14]:
def pick(sentence):
    return sentence.split()

print(pick("I want a cheap hotel in the north"))

['I', 'want', 'a', 'cheap', 'hotel', 'in', 'the', 'north']


In [16]:
def bio_tag(sentence, slots):
    # Step 1 — tokenise the sentence
    tokens = sentence.lower().split()
    
    # Step 2 — start everything as O
    labels = ["O"] * len(tokens)
    
    # Step 3 — for each slot, find its value in the tokens and tag it
    for slot_name, slot_value in slots.items():
        slot_tokens = slot_value.lower().split()
        
        # Find the first word of the slot value in the tokens
        if slot_tokens[0] in tokens:
            idx = tokens.index(slot_tokens[0])
            
            # Tag first word as B-slotname
            labels[idx] = "B-" + slot_name
            
            # Tag remaining words as I-slotname
            for k in range(1, len(slot_tokens)):
                if idx + k < len(tokens):
                    labels[idx + k] = "I-" + slot_name
    
    return tokens, labels

# Test it
tokens, labels = bio_tag(
    "I want a cheap hotel in the north",
    {"pricerange": "cheap", "area": "north"}
)

for token, label in zip(tokens, labels):
    print(f"{token:10} {label}")

i          O
want       O
a          O
cheap      B-pricerange
hotel      O
in         O
the        O
north      B-area


In [17]:
# Extract BIO tagged data from hotel training dialogues
bio_rows = []

for _, dialogue in hotel_train.iterrows():
    turns = dialogue["turns"]
    
    for i, (speaker, utterance, frames) in enumerate(zip(
        turns["speaker"], turns["utterance"], turns["frames"]
    )):
        # Only user turns
        if speaker != 0:
            continue
        
        # Get hotel slots for this turn
        hotel_slots = {}
        for j, state in enumerate(frames["state"]):
            if frames["service"][j] != "hotel":
                continue
            
            names = state["slots_values"]["slots_values_name"]
            values = state["slots_values"]["slots_values_list"]
            
            for name, vals in zip(names, values):
                slot = name.replace("hotel-", "")
                hotel_slots[slot] = vals[0]
        
        if not hotel_slots:
            continue
            
        tokens, labels = bio_tag(utterance, hotel_slots)
        bio_rows.append({"tokens": tokens, "labels": labels})

bio_df = pd.DataFrame(bio_rows)
print(f"BIO tagged turns: {len(bio_df)}")
bio_df.head()

BIO tagged turns: 13731


,tokens,labels
0,"[sounds, good,, could, i, get, that, phone, nu...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-p..."
1,"[yes., can, you, book, it, for, me?]","[O, O, O, O, O, O, O]"
2,"[i, want, to, book, it, for, 2, people, and, 2...","[O, O, O, O, O, O, B-bookstay, O, O, O, O, O, ..."
3,"[guten, tag,, i, am, staying, overnight, in, c...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,"[no,, but, i'd, really, like, to, be, on, the,...","[O, O, O, O, O, O, O, O, O, B-area, O, O, O, O..."


In [18]:
# Count how many of each label we have
from collections import Counter

all_labels = [label for row in bio_df["labels"] for label in row]
label_counts = Counter(all_labels)

for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label:20} {count}")

O                    164775
B-bookstay           1923
B-stars              1794
B-type               1773
I-name               1709
B-bookpeople         1648
B-pricerange         1392
B-area               1164
B-name               1035
B-bookday            802
B-parking            237
B-internet           167


In [19]:
import os
os.makedirs("../data/multiwoz", exist_ok=True)
bio_df.to_json("../data/multiwoz/bio_train.json", orient="records")
print("BIO data saved.")

BIO data saved.


In [20]:
# Create features for each token using a context window
def extract_token_features(tokens, i):
    word = tokens[i].lower()
    
    features = [
        f"word={word}",
        f"is_digit={word.isdigit()}",
        f"prefix2={word[:2]}",
        f"suffix2={word[-2:]}",
        f"prev={tokens[i-1].lower() if i > 0 else 'BOS'}",
        f"next={tokens[i+1].lower() if i < len(tokens)-1 else 'EOS'}"
    ]
    return " ".join(features)

# Test it
test_tokens = ["i", "want", "a", "cheap", "hotel"]
for i in range(len(test_tokens)):
    print(extract_token_features(test_tokens, i))

word=i is_digit=False prefix2=i suffix2=i prev=BOS next=want
word=want is_digit=False prefix2=wa suffix2=nt prev=i next=a
word=a is_digit=False prefix2=a suffix2=a prev=want next=cheap
word=cheap is_digit=False prefix2=ch suffix2=ap prev=a next=hotel
word=hotel is_digit=False prefix2=ho suffix2=el prev=cheap next=EOS


In [21]:
# Build feature and label lists from all BIO tagged turns
X_slot = []
y_slot = []

for _, row in bio_df.iterrows():
    tokens = row["tokens"]
    labels = row["labels"]
    
    for i in range(len(tokens)):
        features = extract_token_features(tokens, i)
        X_slot.append(features)
        y_slot.append(labels[i])

print(f"Total tokens: {len(X_slot)}")
print(f"Total labels: {len(y_slot)}")

Total tokens: 178419
Total labels: 178419


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Split into train and test
X_train_slot, X_test_slot, y_train_slot, y_test_slot = train_test_split(
    X_slot, y_slot, test_size=0.2, random_state=42
)

# Vectorise
slot_vectorizer = TfidfVectorizer()
X_train_vec = slot_vectorizer.fit_transform(X_train_slot)
X_test_vec = slot_vectorizer.transform(X_test_slot)

# Train
slot_classifier = LogisticRegression(class_weight="balanced", max_iter=1000)
slot_classifier.fit(X_train_vec, y_train_slot)

print("Slot classifier trained.")

Slot classifier trained.


In [23]:
y_pred_slot = slot_classifier.predict(X_test_vec)
print(classification_report(y_test_slot, y_pred_slot))

              precision    recall  f1-score   support

      B-area       0.58      0.99      0.74       244
   B-bookday       0.84      1.00      0.91       165
B-bookpeople       0.78      0.97      0.86       348
  B-bookstay       0.72      0.89      0.80       382
  B-internet       0.14      0.77      0.23        31
      B-name       0.25      0.93      0.40       196
   B-parking       0.05      0.15      0.07        39
B-pricerange       0.86      1.00      0.92       284
     B-stars       0.94      0.83      0.89       375
      B-type       0.53      0.98      0.69       368
      I-name       0.21      0.92      0.34       325
           O       1.00      0.92      0.96     32927

    accuracy                           0.92     35684
   macro avg       0.57      0.86      0.65     35684
weighted avg       0.97      0.92      0.94     35684



In [24]:
# Save slot models
with open("../data/models/slot_vectorizer.pkl", "wb") as f:
    pickle.dump(slot_vectorizer, f)

with open("../data/models/slot_classifier.pkl", "wb") as f:
    pickle.dump(slot_classifier, f)

print("Slot models saved.")

Slot models saved.
